# Milestone 5: LightGCN — Tuned

Mirrors the hyperparameter search effort applied to GraphSAGE for a fair comparison.

Pipeline:
1. LR sweep across 4 values (100 epochs each)
2. Tuned config search — embedding dim, layers, neg ratio, epochs
3. Best model selected by Val HR@10
4. Embeddings saved for M6 evaluation

In [1]:
from google.colab import drive
drive.mount('/content/drive')

BASE    = '/content/drive/MyDrive/MLG Project/Project'
OUT_DIR = f'{BASE}/data/processed'
MDL_DIR = f'{BASE}/models'

import os
os.makedirs(MDL_DIR, exist_ok=True)

Mounted at /content/drive


In [2]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'torch-geometric'], check=True)

CompletedProcess(args=['pip', 'install', '-q', 'torch-geometric'], returncode=0)

In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch_geometric.nn import LGConv

import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

Device: cuda


In [4]:
train_pos      = pd.read_csv(f'{OUT_DIR}/train_pos.csv')
val_candidates = pd.read_csv(f'{OUT_DIR}/val_candidates.csv')
test_candidates= pd.read_csv(f'{OUT_DIR}/test_candidates.csv')

print(f"Train edges:         {len(train_pos)}")
print(f"Val candidate rows:  {len(val_candidates)}")
print(f"Test candidate rows: {len(test_candidates)}")

Train edges:         47746
Val candidate rows:  19278
Test candidate rows: 23256


In [5]:
unique_users  = sorted(train_pos['userId'].unique())
unique_movies = sorted(train_pos['movieId'].unique())

user2idx  = {u: i for i, u in enumerate(unique_users)}
movie2idx = {m: i + len(unique_users) for i, m in enumerate(unique_movies)}

num_users  = len(unique_users)
num_movies = len(unique_movies)
num_nodes  = num_users + num_movies

print(f"Users: {num_users}, Movies: {num_movies}, Total nodes: {num_nodes}")

src_u = train_pos['userId'].map(user2idx).values
src_m = train_pos['movieId'].map(movie2idx).values

row = np.concatenate([src_u, src_m])
col = np.concatenate([src_m, src_u])
edge_index = torch.tensor(np.stack([row, col], axis=0), dtype=torch.long).to(DEVICE)
print(f"edge_index shape: {edge_index.shape}")

train_users_t  = torch.tensor(train_pos['userId'].map(user2idx).values, dtype=torch.long).to(DEVICE)
train_movies_t = torch.tensor(
    train_pos['movieId'].map(movie2idx).values - num_users,
    dtype=torch.long).to(DEVICE)

Users: 609, Movies: 6298, Total nodes: 6907
edge_index shape: torch.Size([2, 95492])


## Model & Loss

In [6]:
class LightGCN(nn.Module):
    def __init__(self, num_nodes, emb_dim=64, num_layers=3):
        super().__init__()
        self.emb = nn.Embedding(num_nodes, emb_dim)
        self.convs = nn.ModuleList([LGConv() for _ in range(num_layers)])
        nn.init.xavier_uniform_(self.emb.weight)

    def forward(self, edge_index):
        x = self.emb.weight
        all_embs = [x]
        for conv in self.convs:
            x = conv(x, edge_index)
            all_embs.append(x)
        return torch.stack(all_embs, dim=0).mean(dim=0)

    def get_user_movie_embs(self, edge_index):
        all_emb = self.forward(edge_index)
        return all_emb[:num_users], all_emb[num_users:]


def bpr_loss(user_emb, pos_emb, neg_emb, reg=1e-4):
    pos_scores = (user_emb * pos_emb).sum(dim=1)
    neg_scores = (user_emb * neg_emb).sum(dim=1)
    loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()
    reg_loss = reg * (user_emb.norm(2).pow(2) +
                      pos_emb.norm(2).pow(2) +
                      neg_emb.norm(2).pow(2)) / user_emb.shape[0]
    return loss + reg_loss

## Validation helper (HR@10)

In [7]:
def evaluate_val(model, val_candidates, k=10):
    model.eval()
    with torch.no_grad():
        user_emb, movie_emb = model.get_user_movie_embs(edge_index)
        user_emb  = user_emb.cpu().numpy()
        movie_emb = movie_emb.cpu().numpy()

    hits = []
    for uid, group in val_candidates.groupby('userId'):
        if uid not in user2idx:
            continue
        u_vec = user_emb[user2idx[uid]]
        valid = group[group['movieId'].isin(movie2idx)].copy()
        if valid.empty or valid['label'].sum() == 0:
            continue
        m_indices = valid['movieId'].map(movie2idx).values - num_users
        valid['score'] = movie_emb[m_indices] @ u_vec
        ranked = valid.nlargest(k, 'score')
        hits.append(int(ranked['label'].sum() > 0))
    return float(np.mean(hits)) if hits else 0.0

## Step 1 — Learning Rate Sweep (100 epochs each)

Matches Mirza's GraphSAGE sweep for comparable tuning effort.

In [8]:
def train_model(lr, emb_dim=64, num_layers=3, epochs=100,
                batch_size=2048, neg_ratio=1, reg=1e-4):
    torch.manual_seed(SEED)
    model     = LightGCN(num_nodes, emb_dim, num_layers).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_hr, best_state = 0.0, None

    for epoch in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(train_users_t))
        total_loss, steps = 0.0, 0

        for i in range(0, len(perm), batch_size):
            idx   = perm[i:i + batch_size]
            u_idx = train_users_t[idx]
            p_idx = train_movies_t[idx]

            # Support multiple negatives per positive (mirrors Mirza's neg_ratio)
            if neg_ratio > 1:
                u_idx = u_idx.repeat_interleave(neg_ratio)
                p_idx = p_idx.repeat_interleave(neg_ratio)

            n_idx = torch.randint(0, num_movies, (len(u_idx),), device=DEVICE)

            all_emb = model(edge_index)
            u_emb   = all_emb[u_idx]
            p_emb   = all_emb[num_users + p_idx]
            n_emb   = all_emb[num_users + n_idx]

            loss = bpr_loss(u_emb, p_emb, n_emb, reg=reg)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            steps += 1

        avg_loss = total_loss / steps
        if epoch % 20 == 0 or epoch == 1:
            hr = evaluate_val(model, val_candidates)
            print(f"lr={lr:.0e} | Epoch {epoch:03d}/{epochs} | Loss: {avg_loss:.4f} | Val HR@10: {hr:.4f}")
            if hr > best_hr:
                best_hr = hr
                best_state = {k: v.clone() for k, v in model.state_dict().items()}

    # Load best checkpoint before returning
    if best_state:
        model.load_state_dict(best_state)
    return model, best_hr


learning_rates = [1e-4, 5e-4, 1e-3, 3e-3]
lr_results = {}

for lr in learning_rates:
    print(f"\nTraining LightGCN with lr={lr:.0e}")
    m, hr = train_model(lr=lr, epochs=100)
    lr_results[lr] = {'model': m, 'val_hr10': hr}
    print(f"  -> Best Val HR@10: {hr:.4f}")

print("\n── LR sweep summary ──")
for lr, res in sorted(lr_results.items(), key=lambda x: -x[1]['val_hr10']):
    print(f"lr={lr:.0e} | Best Val HR@10: {res['val_hr10']:.4f}")


Training LightGCN with lr=1e-04
lr=1e-04 | Epoch 001/100 | Loss: 0.6931 | Val HR@10: 0.4118
lr=1e-04 | Epoch 020/100 | Loss: 0.5926 | Val HR@10: 0.8235
lr=1e-04 | Epoch 040/100 | Loss: 0.4614 | Val HR@10: 0.8235
lr=1e-04 | Epoch 060/100 | Loss: 0.4074 | Val HR@10: 0.8235
lr=1e-04 | Epoch 080/100 | Loss: 0.3850 | Val HR@10: 0.8235
lr=1e-04 | Epoch 100/100 | Loss: 0.3790 | Val HR@10: 0.8824
  -> Best Val HR@10: 0.8824

Training LightGCN with lr=5e-04
lr=5e-04 | Epoch 001/100 | Loss: 0.6925 | Val HR@10: 0.8824
lr=5e-04 | Epoch 020/100 | Loss: 0.3778 | Val HR@10: 0.8235
lr=5e-04 | Epoch 040/100 | Loss: 0.3550 | Val HR@10: 0.7647
lr=5e-04 | Epoch 060/100 | Loss: 0.3235 | Val HR@10: 0.7647
lr=5e-04 | Epoch 080/100 | Loss: 0.2834 | Val HR@10: 0.7647
lr=5e-04 | Epoch 100/100 | Loss: 0.2562 | Val HR@10: 0.7647
  -> Best Val HR@10: 0.8824

Training LightGCN with lr=1e-03
lr=1e-03 | Epoch 001/100 | Loss: 0.6904 | Val HR@10: 0.8235
lr=1e-03 | Epoch 020/100 | Loss: 0.3442 | Val HR@10: 0.8235
lr=1e

## Step 2 — Tuned Config Search

Bigger embedding dim, more layers, more epochs, higher neg ratio — equivalent to Mirza's tuned configs.

In [9]:
search_space = [
    {"lr": 1e-3, "emb_dim": 128, "num_layers": 3, "epochs": 220, "neg_ratio": 3, "reg": 1e-4},
    {"lr": 3e-3, "emb_dim": 128, "num_layers": 3, "epochs": 220, "neg_ratio": 3, "reg": 1e-4},
    {"lr": 5e-3, "emb_dim": 128, "num_layers": 4, "epochs": 220, "neg_ratio": 3, "reg": 1e-5},
    {"lr": 3e-3, "emb_dim": 192, "num_layers": 4, "epochs": 240, "neg_ratio": 4, "reg": 1e-5},
]

tuned_results = []
best_tuned_hr   = max(r['val_hr10'] for r in lr_results.values())
best_tuned_model = list(lr_results.values())[0]['model']  # will be overwritten

for cfg in search_space:
    print(f"\nTuned config: {cfg}")
    m, hr = train_model(**cfg)
    tuned_results.append({**cfg, 'val_hr10': hr, 'model': m})
    print(f"  -> Best Val HR@10: {hr:.4f}")
    if hr > best_tuned_hr:
        best_tuned_hr   = hr
        best_tuned_model = m

print(f"\nBest tuned Val HR@10: {best_tuned_hr:.4f}")


Tuned config: {'lr': 0.001, 'emb_dim': 128, 'num_layers': 3, 'epochs': 220, 'neg_ratio': 3, 'reg': 0.0001}
lr=1e-03 | Epoch 001/220 | Loss: 0.6874 | Val HR@10: 0.8235
lr=1e-03 | Epoch 020/220 | Loss: 0.2819 | Val HR@10: 0.8235
lr=1e-03 | Epoch 040/220 | Loss: 0.2139 | Val HR@10: 0.8235
lr=1e-03 | Epoch 060/220 | Loss: 0.1687 | Val HR@10: 0.8235
lr=1e-03 | Epoch 080/220 | Loss: 0.1380 | Val HR@10: 0.8235
lr=1e-03 | Epoch 100/220 | Loss: 0.1158 | Val HR@10: 0.8824
lr=1e-03 | Epoch 120/220 | Loss: 0.1004 | Val HR@10: 0.8824
lr=1e-03 | Epoch 140/220 | Loss: 0.0883 | Val HR@10: 0.8824
lr=1e-03 | Epoch 160/220 | Loss: 0.0798 | Val HR@10: 0.9412
lr=1e-03 | Epoch 180/220 | Loss: 0.0740 | Val HR@10: 0.9412
lr=1e-03 | Epoch 200/220 | Loss: 0.0677 | Val HR@10: 0.8824
lr=1e-03 | Epoch 220/220 | Loss: 0.0629 | Val HR@10: 0.8824
  -> Best Val HR@10: 0.9412

Tuned config: {'lr': 0.003, 'emb_dim': 128, 'num_layers': 3, 'epochs': 220, 'neg_ratio': 3, 'reg': 0.0001}
lr=3e-03 | Epoch 001/220 | Loss: 0.6

## Step 3 — Save best model embeddings

In [10]:
model = best_tuned_model
model.eval()

with torch.no_grad():
    user_emb_final, movie_emb_final = model.get_user_movie_embs(edge_index)

user_emb_np  = user_emb_final.cpu().numpy()
movie_emb_np = movie_emb_final.cpu().numpy()

np.save(f'{OUT_DIR}/user_emb.npy',  user_emb_np)
np.save(f'{OUT_DIR}/movie_emb.npy', movie_emb_np)

pd.Series(user2idx).rename_axis('userId').rename('idx').reset_index().to_csv(
    f'{OUT_DIR}/user2idx.csv', index=False)
pd.Series(movie2idx).rename_axis('movieId').rename('idx').reset_index().to_csv(
    f'{OUT_DIR}/movie2idx.csv', index=False)

torch.save(model.state_dict(), f'{MDL_DIR}/lightgcn_best_tuned.pt')

print(f"User embeddings:  {user_emb_np.shape}")
print(f"Movie embeddings: {movie_emb_np.shape}")
print("All files saved — ready for M6.")

User embeddings:  (609, 128)
Movie embeddings: (6298, 128)
All files saved — ready for M6.
